# 🔶 Figure 4A DNA level comparison

1. Y-axis: AUC/AUPR
2. X-axis: Different features
3. Each group on X-axis: Transfer (Noncoding pretrained), Transfer (Missense pretrained), Baseline (No pretrained), 10 replicates per experiment. If one of the Transfer groups outperforms Baseline, provide statistical significance


NOTE: Encoder: Unified ResNet, using Conv to capture local motifs

## (1) Prepare data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel


# Basic configuration
sig_color = '#333'
features = ['onehot-DNA', 'NT', 'HyenaDNA', 'SpliceBERT', 'GPN-MSA',]
sources = ['noncoding', 'missense', ]
feature_labels = {
    'onehot-DNA': 'One-hot (DNA)',
    'NT': 'NT',
    'HyenaDNA': 'HyenaDNA',
    'SpliceBERT': 'SpliceBERT',
    'GPN-MSA': 'GPN-MSA',
}
colors = {
    'Transfer_noncoding': '#4a70a3',
    'Transfer_missense': "#8ab7c6",
    'Baseline': '#e1ddd6'
}
compared_metric = 'AUPR' # AUC | AUPR

# Read data
all_records = []
for feature in features:
    scratch_dict = {}
    # First collect scratch from both sources
    scratch_tables = []
    for source in sources:
        path = f"./pretrained/{source}-{feature}/Finetune.ResNet1D.metric.summary.txt"
        df = pd.read_csv(path, sep='\t')
        df['feature'] = feature
        df['source'] = source
        # Save scratch
        scratch_tables.append(
            df[df['group'] == 'scratch']
        )
        # Save finetune
        ft = df[df['group'] == 'finetune'].copy()
        ft['plot_group'] = f'Transfer_{source}'
        all_records.append(ft)

    # baseline = average of scratch from both sources
    scratch_all = pd.concat(scratch_tables)
    baseline = (
        scratch_all
        .groupby(['seed'])
        .agg({compared_metric: 'mean'})
        .reset_index()
    )
    baseline['feature'] = feature
    baseline['plot_group'] = 'Baseline'
    all_records.append(baseline)

# Merge
plot_df = pd.concat(all_records, ignore_index=True)

## (2) Plot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel


# Basic configuration
sig_color = '#333'  # Significance color marker #cc342d
features = ['onehot-DNA', 'NT', 'HyenaDNA', 'SpliceBERT', 'GPN-MSA',]
sources = ['noncoding', 'missense', ]
feature_labels = {
    'onehot-DNA': 'One-hot (DNA)',
    'NT': 'NT',
    'HyenaDNA': 'HyenaDNA',
    'SpliceBERT': 'SpliceBERT',
    'GPN-MSA': 'GPN-MSA',
}
colors = {
    'Transfer_noncoding': '#4a70a3',
    'Transfer_missense': "#8ab7c6",
    'Baseline': '#e1ddd6'
}
compared_metric = 'AUPR'

# Read data
all_records = []
for feature in features:
    scratch_dict = {}
    # First collect scratch from both sources
    scratch_tables = []
    for source in sources:
        path = f"./pretrained_weight/{source}-{feature}/Finetune.ResNet1D.metric.summary.txt"
        df = pd.read_csv(path, sep='\t')
        df['feature'] = feature
        df['source'] = source
        # Save scratch
        scratch_tables.append(
            df[df['group'] == 'scratch']
        )
        # Save finetune
        ft = df[df['group'] == 'finetune'].copy()
        ft['plot_group'] = f'Transfer_{source}'
        all_records.append(ft)

    # baseline = average of scratch from both sources
    scratch_all = pd.concat(scratch_tables)
    baseline = (
        scratch_all
        .groupby(['seed'])
        .agg({compared_metric: 'mean'})
        .reset_index()
    )
    baseline['feature'] = feature
    baseline['plot_group'] = 'Baseline'
    all_records.append(baseline)

# Merge
plot_df = pd.concat(all_records, ignore_index=True)

# Statistical test
stat_results = []
for feature in features:
    sub = plot_df[plot_df['feature'] == feature]
    baseline = sub[sub['plot_group'] == 'Baseline']
    for source in sources:
        transfer = sub[
            sub['plot_group'] == f'Transfer_{source}'
        ]
        merged = pd.merge(
            transfer[['seed', compared_metric]],
            baseline[['seed', compared_metric]],
            on='seed',
            suffixes=('_transfer', '_baseline')
        )
        t, p = ttest_rel(
            merged[f'{compared_metric}_transfer'],
            merged[f'{compared_metric}_baseline']
        )
        stat_results.append({
            'feature': feature,
            'source': source,
            'p': p
        })
stat_df = pd.DataFrame(stat_results)

# Plotting
plt.figure(figsize=(10.5, 4.5))

order = [
    'Transfer_noncoding',
    'Transfer_missense',
    'Baseline'
]

# boxplot
ax = sns.boxplot(
    data=plot_df,
    x='feature',
    y=compared_metric,
    hue='plot_group',
    order=features,
    hue_order=order,
    palette=colors,
    # width=0.6,
    width=0.8,
    saturation=1,  # saturation
    linewidth=0.7,  # line thickness
    # fliersize=0  # Do not show outliers
    flierprops={  # Control outlier format
        'marker': 'o',        # Marker shape: 'o', 's', '^', 'D', '+', 'x', etc.
        'markersize': 4,      # Size
        'markerfacecolor': 'None',  # Fill color
        'markeredgecolor': sig_color,  # Border color
        'markeredgewidth': 1,  # Border thickness
        'alpha': 0.7,         # Transparency
    }
)
# Set legends
from matplotlib.lines import Line2D
markersize = 12
legend_elements = [
    Line2D([0], [0],
           marker='o',
           linestyle='None',              # Remove border from legend
           label='Transfer (Noncoding)',
           markerfacecolor=colors['Transfer_noncoding'],
           markeredgecolor='none',        
           markeredgewidth=0,             
           markersize=markersize),

    Line2D([0], [0],
           marker='o',
           linestyle='None',
           label='Transfer (Missense)',
           markerfacecolor=colors['Transfer_missense'],
           markeredgecolor='none',
           markeredgewidth=0,
           markersize=markersize),

    Line2D([0], [0],
           marker='o',
           linestyle='None',
           label='Baseline (No pretrained)',
           markerfacecolor=colors['Baseline'],
           markeredgecolor='none',
           markeredgewidth=0,
           markersize=markersize),
]
ax.legend(
    handles=legend_elements,
    # loc='upper center',
    # bbox_to_anchor=(0.5, 1),
    # ncol=3,
    loc='lower right',
    bbox_to_anchor=(1, -0.02),
    ncol=1,
    frameon=False,
    fontsize=12,
)

# Reset x labels
ax.set_xticklabels(
    [feature_labels[f] for f in features], fontsize=12
)

# Significance markers
y_max = plot_df[compared_metric].max()
offset = 0.01
for i, feature in enumerate(features):
    for j, source in enumerate(sources):
        row = stat_df[
            (stat_df['feature'] == feature) &
            (stat_df['source'] == source)
        ]
        p = row['p'].values[0]
        if p < 0.05:
            # print(f"{feature} {source} has significant improvement: {p}")
            # Determine significance direction
            transfer_mean = plot_df[(plot_df['feature'] == feature) & (plot_df['source'] == source)][compared_metric].mean()
            baseline_mean = plot_df[(plot_df['feature'] == feature) & (plot_df['plot_group'] == 'Baseline')][compared_metric].mean()
            if transfer_mean > baseline_mean:
                arrow = '↑'
            else:
                arrow = '↓'
            x1 = i - 0.22 + j * 0.22
            x2 = i + 0.22
            # print(f"from {x1} to {x2}")
            if j == 0:
                y = y_max + offset * (j + 1)
            else:
                y = y_max - offset * (j + 1.5) # NOTE AUC
                # y = y_max - offset * (j + 2) # NOTE AUPR
            plt.plot(
                [x1, x1, x2, x2],
                [y, y + offset,
                 y + offset, y],
                lw=0.5,
                c=sig_color
            )
            if p < 0.001:
                star = '***'
            elif p < 0.01:
                star = '**'
            else:
                star = '*'
            plt.text(
                (x1 + x2) / 2,
                y + offset + 0.005,
                star + arrow, # NOTE Added direction (improvement/decline)
                ha='center',
                color=sig_color,
                fontsize=14,
                fontweight='100'  # Font weight
            )

# Styling
plt.ylabel(compared_metric, fontsize=13)
# plt.xlabel("Features", fontsize=13)
plt.xlabel("", fontsize=13)

# Uniform tick scale
from matplotlib.ticker import FormatStrFormatter
step = 0.1
y_min = 0.45
# y_max = 0.95 # NOTE AUC
y_max = 1 # NOTE AUPR
plt.ylim(y_min, y_max)
# Set ticks
# yticks = np.arange(y_min, y_max + step, step)
# plt.yticks(yticks)
# Keep two decimal places
plt.gca().yaxis.set_major_formatter(
    FormatStrFormatter('%.2f')
)
# Set font size
plt.tick_params(axis='y', labelsize=12) 

# Borders
for spine in ax.spines.values():
    spine.set_linewidth(0.5)      # Thickness
    spine.set_color('#000')       # Color
plt.tight_layout()
plt.show()

# 🔶 Figure 4B RNA level comparison


## (1) Prepare data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel

# Basic configuration
sig_color = '#333'
features = ['onehot-RNA', 'RiNALMo', 'ERNIE-RNA']
feature_labels = {
    'onehot-RNA': 'One-hot (RNA)',
    # 'SpliceBERT': 'SpliceBERT',
    'RiNALMo': 'RiNALMo',
    'ERNIE-RNA': 'ERNIE-RNA',
}
colors = {
    'Transfer_missense': '#8ab7c6',
    'Baseline': '#e1ddd6'
}
compared_metric = 'AUPR' # AUC | AUPR
groups = [
    'Transfer_missense',
    'Baseline'
]

# Read data
all_records = []
for feature in features:
    path = f"./pretrained_weight/missense-{feature}/Finetune.PositionTransformer.metric.summary.txt"
    df = pd.read_csv(path, sep='\t')
    # Transfer
    transfer = df[df['group'] == 'finetune'].copy()
    transfer['plot_group'] = 'Transfer_missense'
    transfer['feature'] = feature
    # Baseline
    baseline = df[df['group'] == 'scratch'].copy()
    baseline['plot_group'] = 'Baseline'
    baseline['feature'] = feature
    all_records.append(transfer)
    all_records.append(baseline)
plot_df = pd.concat(all_records, ignore_index=True)

# Statistical test (paired t-test)
stat_results = []
for feature in features:
    sub = plot_df[plot_df['feature'] == feature]
    transfer = sub[sub['plot_group'] == 'Transfer_missense']
    baseline = sub[sub['plot_group'] == 'Baseline']

    merged = pd.merge(
        transfer[['seed', compared_metric]],
        baseline[['seed', compared_metric]],
        on='seed',
        suffixes=('_transfer', '_baseline')
    )
    t, p = ttest_rel(
        merged[f'{compared_metric}_transfer'],
        merged[f'{compared_metric}_baseline']
    )
    stat_results.append({
        'feature': feature,
        'p': p
    })
stat_df = pd.DataFrame(stat_results)

## (2) Plot

In [ ]:
# Plotting
plt.figure(figsize=(5.5, 4))
# boxplot
ax = sns.boxplot(
    data=plot_df,
    x='feature',
    y=compared_metric,
    hue='plot_group',
    order=features,
    hue_order=groups,
    palette=colors,
    width=0.6,       # Control within-group spacing
    linewidth=0.7,
    # fliersize=0,
    flierprops={  # Control outlier format
        'marker': 'o',        # Marker shape: 'o', 's', '^', 'D', '+', 'x', etc.
        'markersize': 4,      # Size
        'markerfacecolor': 'None',  # Fill color
        'markeredgecolor': sig_color,  # Border color
        'markeredgewidth': 1,  # Border thickness
        'alpha': 0.7,         # Transparency
    }
)


# Unified legend
from matplotlib.lines import Line2D
markersize = 12
legend_elements = [
    Line2D([0], [0],
           marker='o',
           linestyle='None',
           label='Transfer (Missense)',
           markerfacecolor=colors['Transfer_missense'],
           markeredgecolor='none',
           markeredgewidth=0,
           markersize=markersize),

    Line2D([0], [0],
           marker='o',
           linestyle='None',
           label='Baseline (No pretrained)',
           markerfacecolor=colors['Baseline'],
           markeredgecolor='none',
           markeredgewidth=0,
           markersize=markersize),
]
ax.legend(
    handles=legend_elements,
    loc='lower left',
    ncol=1,
    frameon=False,
    fontsize=12,
)

# Reset x labels
ax.set_xticklabels(
    [feature_labels[f] for f in features], fontsize=12
)

# Significance markers
hue_offsets = {
    'Transfer_missense': -0.20,
    'Baseline': 0.20
}
y_max = plot_df[compared_metric].max()
for i, feature in enumerate(features):
    # Current feature data
    sub = plot_df[plot_df['feature'] == feature]
    transfer = sub[sub['plot_group'] == 'Transfer_missense']
    baseline = sub[sub['plot_group'] == 'Baseline']

    # paired t-test significance test
    merged = pd.merge(
        transfer[['seed', compared_metric]],
        baseline[['seed', compared_metric]],
        on='seed',
        suffixes=('_transfer', '_baseline')
    )
    t, p = ttest_rel(
        merged[f'{compared_metric}_transfer'],
        merged[f'{compared_metric}_baseline']
    )

    # Draw significance marker
    if p < 0.05:
        # Determine significance direction (whether transfer improves or decreases performance)
        transfer_mean = transfer[compared_metric].mean()
        baseline_mean = baseline[compared_metric].mean()
        if transfer_mean > baseline_mean:
            arrow = '↑'
        else:
            arrow = '↓'

        # Star significance levels
        if p < 0.001:
            star = '***'
        elif p < 0.01:
            star = '**'
        else:
            star = '*'

        # Line positions
        x_transfer = i + hue_offsets['Transfer_missense']
        x_baseline = i + hue_offsets['Baseline']
        y = y_max + 0.01
        plt.plot(
            [x_transfer, x_transfer,
             x_baseline, x_baseline],
            [y, y+0.005, y+0.005, y],
            lw=0.5,
            c=sig_color
        )

        # Draw marker
        plt.text(
            (x_transfer + x_baseline) / 2,
            y + 0.01,
            star + arrow,
            ha='center',
            color=sig_color,
            fontsize=14,
            fontweight='100'  # Font weight
        )

# Styling
# x-label modification
ax.set_xticklabels([feature_labels[f] for f in features], fontsize=12)
plt.ylabel(compared_metric, fontsize=13)
# plt.xlabel("Features", fontsize=13)
plt.xlabel("", fontsize=13)

# Uniform tick scale
from matplotlib.ticker import FormatStrFormatter, MultipleLocator
step = 0.05

y_min = 0.6  # AUC
y_max = 0.85  # AUC 

# y_min = 0.65  # AUPR
# y_max = 0.9  # AUPR
plt.ylim(y_min, y_max)
# Keep two decimal places
plt.gca().yaxis.set_major_locator(MultipleLocator(step))
plt.gca().yaxis.set_major_formatter(
    FormatStrFormatter('%.2f')
)
# Set font size
plt.tick_params(axis='y', labelsize=12) 

# Borders
for spine in ax.spines.values():
    spine.set_linewidth(0.5)      # Thickness
    spine.set_color('#000')       # Color
plt.tight_layout()
plt.show()

# 🔶 Figure 4D Protein level comparison

## (1) Prepare data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel

# Basic configuration
sig_color = '#333'
features = ['onehot-protein', 'CaLM', 'ISM']
feature_labels = {
    'onehot-protein': 'One-hot (protein)',
    'CaLM': 'CaLM',
    'ISM': 'ISM',
}
colors = {
    'Transfer_missense': '#8ab7c6',
    'Baseline': '#e1ddd6'
}
compared_metric = 'AUPR'
groups = [
    'Transfer_missense',
    'Baseline'
]

# Read data
all_records = []
for feature in features:
    path = f"./pretrained_weight/missense-{feature}/Finetune.PositionTransformer.metric.summary.txt"
    df = pd.read_csv(path, sep='\t')
    # Transfer
    transfer = df[
        df['group'] == 'finetune'
    ].copy()
    transfer['plot_group'] = 'Transfer_missense'
    transfer['feature'] = feature
    # Baseline
    baseline = df[
        df['group'] == 'scratch'
    ].copy()
    baseline['plot_group'] = 'Baseline'
    baseline['feature'] = feature

    all_records.append(transfer)
    all_records.append(baseline)
plot_df = pd.concat(all_records, ignore_index=True)

## (2) Plot

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel

# Basic configuration
sig_color = '#333'
features = ['onehot-protein', 'CaLM', 'ISM']
feature_labels = {
    'onehot-protein': 'One-hot (protein)',
    'CaLM': 'CaLM',
    'ISM': 'ISM',
}
colors = {
    'Transfer_missense': '#8ab7c6',
    'Baseline': '#e1ddd6'
}
compared_metric = 'AUPR'
groups = [
    'Transfer_missense',
    'Baseline'
]

# Read data
all_records = []
for feature in features:
    path = f"./pretrained_weight/missense-{feature}/Finetune.PositionTransformer.metric.summary.txt"
    df = pd.read_csv(path, sep='\t')
    # Transfer
    transfer = df[
        df['group'] == 'finetune'
    ].copy()
    transfer['plot_group'] = 'Transfer_missense'
    transfer['feature'] = feature
    # Baseline
    baseline = df[
        df['group'] == 'scratch'
    ].copy()
    baseline['plot_group'] = 'Baseline'
    baseline['feature'] = feature

    all_records.append(transfer)
    all_records.append(baseline)
plot_df = pd.concat(all_records, ignore_index=True)

# Statistical test (paired t-test)
stat_results = []
for feature in features:
    sub = plot_df[
        plot_df['feature'] == feature
    ]
    transfer = sub[
        sub['plot_group'] == 'Transfer_missense'
    ]
    baseline = sub[
        sub['plot_group'] == 'Baseline'
    ]
    merged = pd.merge(
        transfer[['seed', compared_metric]],
        baseline[['seed', compared_metric]],
        on='seed',
        suffixes=('_transfer', '_baseline')
    )
    t, p = ttest_rel(
        merged[f'{compared_metric}_transfer'],
        merged[f'{compared_metric}_baseline']
    )
    stat_results.append({
        'feature': feature,
        'p': p
    })
stat_df = pd.DataFrame(stat_results)

# Plotting
plt.figure(figsize=(5.5, 4))

ax = sns.boxplot(
    data=plot_df,
    x='feature',
    y=compared_metric,
    hue='plot_group',
    order=features,
    hue_order=groups,
    palette=colors,
    width=0.6,       # Control within-group spacing
    # fliersize=0,
    linewidth=0.7,
    flierprops={  # Control outlier format
        'marker': 'o',       
        'markersize': 4,      # Size
        'markerfacecolor': 'None',  # Fill color
        'markeredgecolor': sig_color,  # Border color
        'markeredgewidth': 1,  # Border thickness
        'alpha': 0.7,         # Transparency
    }
)

# Unified legend
from matplotlib.lines import Line2D
markersize = 12
legend_elements = [
    Line2D([0], [0],
           marker='o',
           linestyle='None',
           label='Transfer (Missense)',
           markerfacecolor=colors['Transfer_missense'],
           markeredgecolor='none',
           markeredgewidth=0,
           markersize=markersize),

    Line2D([0], [0],
           marker='o',
           linestyle='None',
           label='Baseline (No pretrained)',
           markerfacecolor=colors['Baseline'],
           markeredgecolor='none',
           markeredgewidth=0,
           markersize=markersize),
]
ax.legend(
    handles=legend_elements,
    # loc='lower left',
    # ncol=1,
    loc='lower left',
    ncol=1,
    frameon=False,
    fontsize=12,
    columnspacing=0.5
)

# x-label modification
ax.set_xticklabels([feature_labels[f] for f in features], fontsize=12)

# Significance markers
hue_offsets = {
    'Transfer_missense': -0.20,
    'Baseline': 0.20
}
y_max = plot_df[compared_metric].max()
for i, feature in enumerate(features):
    # Current feature data
    sub = plot_df[plot_df['feature'] == feature]
    transfer = sub[sub['plot_group'] == 'Transfer_missense']
    baseline = sub[sub['plot_group'] == 'Baseline']

    # paired t-test significance test
    merged = pd.merge(
        transfer[['seed', compared_metric]],
        baseline[['seed', compared_metric]],
        on='seed',
        suffixes=('_transfer', '_baseline')
    )
    t, p = ttest_rel(
        merged[f'{compared_metric}_transfer'],
        merged[f'{compared_metric}_baseline']
    )

    # Draw significance marker
    if p < 0.05:
        # Determine significance direction (whether transfer improves or decreases performance)
        transfer_mean = transfer[compared_metric].mean()
        baseline_mean = baseline[compared_metric].mean()
        if transfer_mean > baseline_mean:
            arrow = '↑'
        else:
            arrow = '↓'

        # Star significance levels
        if p < 0.001:
            star = '***'
        elif p < 0.01:
            star = '**'
        else:
            star = '*'

        # Line positions
        x_transfer = i + hue_offsets['Transfer_missense']
        x_baseline = i + hue_offsets['Baseline']
        y = y_max + 0.01
        plt.plot(
            [x_transfer, x_transfer,
             x_baseline, x_baseline],
            [y, y+0.005, y+0.005, y],
            lw=0.5,
            c=sig_color
        )

        # Draw marker
        plt.text(
            (x_transfer + x_baseline) / 2,
            y + 0.01,
            star + arrow,
            ha='center',
            color=sig_color,
            fontsize=14,
            fontweight='100'  # Font weight
        )

# Styling
plt.ylabel(compared_metric, fontsize=13)
# plt.xlabel("Features", fontsize=13)
plt.xlabel("", fontsize=13)

# Uniform tick scale
from matplotlib.ticker import FormatStrFormatter, MultipleLocator
step = 0.05

# y_min = 0.65  # AUC
# y_max = 0.80  # AUC

y_min = 0.6  # AUPR
y_max = 0.81  # AUPR

plt.ylim(y_min, y_max)
# Keep two decimal places
plt.gca().yaxis.set_major_locator(MultipleLocator(step))
plt.gca().yaxis.set_major_formatter(
    FormatStrFormatter('%.2f')
)
# Set font size
plt.tick_params(axis='y', labelsize=12) 

# Borders
for spine in ax.spines.values():
    spine.set_linewidth(0.5)      # Thickness
    spine.set_color('#000')       # Color
plt.tight_layout()
plt.tight_layout()
plt.show()

# 🔶 Figure 4E Foundation model zero-shot

NOTE: GPN-MSA provides the prediction for all SNVs, we use its pre-computed scores.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


# Zero-shot result
results = {
    "GPN-MSA":     {"AUC": 0.7535, "AUPR": 0.7906},
    "CaLM":        {"AUC": 0.6380, "AUPR": 0.6580},
    "ERNIE-RNA":   {"AUC": 0.5258, "AUPR": 0.5585},
    "SpliceBERT":  {"AUC": 0.6263, "AUPR": 0.6221},
    "NT":          {"AUC": 0.5000, "AUPR": 0.5000},
    "RiNALMo":     {"AUC": 0.5516, "AUPR": 0.5724},
}


# config
models = list(results.keys())

auc_list = [results[m]["AUC"] for m in models]
aupr_list = [results[m]["AUPR"] for m in models]

colors = ["#3b6eb2", "#c2ab83", "#80bcd0", "#e8e3db", "#d95f5f", "#7a6ff0"]
markers = ["o", "s", "D", "^", "v", "P"]


# plot
groups = ["AUC", "AUPR"]
x = np.arange(len(groups))

offsets = np.linspace(-0.2, 0.2, len(models))

fig, ax = plt.subplots(figsize=(5, 5))

for i, model in enumerate(models):

    y_vals = [auc_list[i], aupr_list[i]]

    ax.scatter(
        x + offsets[i],
        y_vals,
        color=colors[i % len(colors)],
        marker=markers[i % len(markers)],
        s=110,
        label=model,
        zorder=3
    )

    # value annotation
    for j, val in enumerate(y_vals):
        ax.text(
            x[j] + offsets[i],
            val + 0.012,
            f"{val:.3f}",
            ha="center",
            va="bottom",
            fontsize=10
        )


# axis settings
ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylabel("Score", fontsize=12)

ax.set_ylim(0.45, 0.85)

ax.legend(
    ncols=3,
    frameon=False,
    bbox_to_anchor=(1.03, 1.25),
    fontsize=11
)

ax.grid(axis="y", linestyle=":", alpha=0.4)

mono_color = "#222"
for spine in ax.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color(mono_color)

ax.tick_params(
    axis="both",
    labelsize=12,
    width=1.2,
    color=mono_color,
    labelcolor=mono_color
)

plt.tight_layout()
plt.show()

# 🔶 Figure 4F Model Ablation Result

## (1) Prepare data

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")

df = pd.read_csv("./result/section_4/model_ablation/GPN_CaLM_NewBio/metrics.txt", sep='\t')

mapping = {
    'dna': 'GPN-MSA',
    'rna': 'CaLM',
    'bio': 'Handcrafted',
    'dna+rna': 'GPN-MSA + CaLM',
    'rna+bio': 'CaLM + Handcrafted',
    'dna+bio': 'GPN-MSA + Handcrafted'
}
# order control
order = [
    'CaLM',
    'GPN-MSA',
    'Handcrafted',
    'GPN-MSA + CaLM',
    'CaLM + Handcrafted',
    'GPN-MSA + Handcrafted',
    'GPN-MSA + CaLM + Handcrafted',
]

df['submodule'] = df['submodule'].map(mapping)

# whole model 
df2 = pd.read_csv("./result/section_2/GPN_CaLM_NewBio/metrics.txt", sep='\t')
df_all = df2[(df2['TRANSFER'] == True) & (df2['pct'] == 0.8)]
df_all['submodule'] = ['GPN-MSA + CaLM + Handcrafted'] * len(df_all)
df_all = df_all[['SEED', 'submodule', 'AUC', 'AUPR']]

# merge
df = pd.concat([df, df_all], ignore_index=True)

# groupby 
stats_auc = df.groupby('submodule')['AUC'].agg(['mean', 'std', 'count']).reset_index()
stats_auc['sem'] = stats_auc['std'] / np.sqrt(stats_auc['count'])
stats_auc['metric'] = 'AUC'

stats_aupr = df.groupby('submodule')['AUPR'].agg(['mean', 'std', 'count']).reset_index()
stats_aupr['sem'] = stats_aupr['std'] / np.sqrt(stats_aupr['count'])
stats_aupr['metric'] = 'AUPR'

stats_all = pd.concat([stats_auc, stats_aupr], ignore_index=True)

# control order
stats_all['submodule'] = pd.Categorical(stats_all['submodule'], categories=order, ordered=True)
stats_all = stats_all.sort_values(['metric', 'submodule'])

## (2) Plot

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import ttest_rel
from matplotlib.lines import Line2D

# =========================
# config
# =========================
colors = ["#d95f5f", "#c2ab83", "#e0d9ce", "#c6c9e8", "#ddc5c5", "#80bcd0", '#3b6eb2']
metric_list = ['AUC', 'AUPR']
group_x = np.arange(len(metric_list)) 
n_sub = len(order)
offsets = np.linspace(-0.3, 0.3, n_sub)
bar_width = 0.07
plt.figure(figsize=(16, 4))
ax = plt.gca()
x_map = {}

# =========================
# 1. bar plot
# =========================
for i, sub in enumerate(order):
    sub_df = stats_all[stats_all['submodule'] == sub]
    means, sems = [], []
    for m in metric_list:
        row = sub_df[sub_df['metric'] == m]
        means.append(row['mean'].values[0])
        sems.append(row['sem'].values[0])

    x = group_x + offsets[i]
    ax.bar(
        x,
        means,
        width=bar_width,
        color=colors[i],
        yerr=sems,
        capsize=4,
        linewidth=0.4,
        error_kw=dict(ecolor='#333', linewidth=0.8)
    )

    for j, m in enumerate(metric_list):
        x_map[(sub, m)] = x[j]


# =========================
# 2. paired t-test
# =========================
pairs = [
    ("Handcrafted", "CaLM + Handcrafted"),
    ("Handcrafted", "GPN-MSA + Handcrafted"),
    ("GPN-MSA", "GPN-MSA + CaLM"),
    ("CaLM + Handcrafted", "GPN-MSA + CaLM + Handcrafted")
]

stat_results = []

for base, trans in pairs:
    for metric in metric_list:

        base_df = df[df['submodule'] == base]
        trans_df = df[df['submodule'] == trans]

        merged = pd.merge(
            trans_df[['SEED', metric]],
            base_df[['SEED', metric]],
            on='SEED',
            suffixes=('_t', '_b')
        )

        if len(merged) == 0:
            continue

        t, p = ttest_rel(
            merged[f'{metric}_t'],
            merged[f'{metric}_b']
        )

        stat_results.append({
            'base': base,
            'trans': trans,
            'metric': metric,
            'p': p,
            'mean_diff': merged[f'{metric}_t'].mean() -
                         merged[f'{metric}_b'].mean()
        })

stat_df = pd.DataFrame(stat_results)


# =========================
# 3. significance plotting
# =========================
y_max = stats_all['mean'].max()
offset_y = 0.01

for k, row in enumerate(stat_df.itertuples()):

    if row.p >= 0.05:
        continue

    x1 = x_map[(row.base, row.metric)]
    x2 = x_map[(row.trans, row.metric)]

    y = y_max + offset_y * (k + 1.5) - 0.05

    # bracket
    ax.plot(
        [x1, x1, x2, x2],
        [y, y + offset_y, y + offset_y, y],
        lw=0.5,
        c='#333'
    )

    # significance level
    if row.p < 0.001:
        star = '***'
    elif row.p < 0.01:
        star = '**'
    else:
        star = '*'

    arrow = '↑' if row.mean_diff > 0 else '↓'

    ax.text(
        (x1 + x2) / 2,
        y + offset_y * 1.2,
        star + arrow,
        ha='center',
        fontsize=18,
        color='#333',
        fontweight='100'
    )


# =========================
# 4. axis
# =========================
plt.xticks(group_x, metric_list)
plt.ylabel('Score', fontsize=13)

# plt.grid(axis='y', linestyle='--', alpha=0.4)


# =========================
# 5. legend (custom circles)
# =========================
handles = []
labels = []

for i, sub in enumerate(order):
    handles.append(
        Line2D(
            [0], [0],
            marker='o',
            linestyle='none',
            markerfacecolor=colors[i],
            markeredgecolor='none',
            markersize=17
        )
    )
    labels.append(sub)

plt.legend(
    handles,
    labels,
    ncol=1,
    frameon=False,
    bbox_to_anchor=[1.35, 0],
    loc='lower right',
    fontsize=12,
    labelspacing=1.5
)


# =========================
# 6. style
# =========================
plt.tick_params(axis='both', labelsize=12)
plt.ylim(0.7, 1.05)
plt.yticks(np.arange(0.7, 1.01, 0.05))

for spine in ax.spines.values():
    spine.set_linewidth(0.5)
    spine.set_color('#000')

plt.tight_layout()
plt.show()